# Walkthrough: evidence-grounded review of financial statement records

This notebook reads the committed results and walks through one company, one
flag, its evidence record, its query text and the audit log. Set `EGR_QUICK=1`
before starting the kernel to read `results-quick/` instead of `results/`.
Nothing here recomputes a result; everything comes from the tables the
pipeline wrote.

In [1]:
import json

import pandas as pd

from egr import config
from egr.auditlog import verify
from egr.faithfulness import check_text
from egr.querytext import template

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
results = config.RESULTS_DIR
tables = results / "tables"
summary = json.loads((results / "summary.json").read_text(encoding="utf-8"))
print("results from", results.name)
print(
    "held-out companies:",
    summary["heldout_companies"],
    " held-out tested cells:",
    summary["heldout_tested_cells"],
)
print(
    "share of held-out cells with a plausibility score:", round(summary["heldout_scored_share"], 3)
)

results from results
held-out companies: 191  held-out tested cells: 168777
share of held-out cells with a plausibility score: 0.973


## 1. The data

Companies, periods and the two kinds of real-error labels, from the prepare step.

In [2]:
prep = summary["prepare"]
pd.Series(prep).drop("companies_dropped").to_frame("value")

,value
companies_in_universe,400
companies_kept,382
development,191
heldout,191
cells,347190
tested_cells,340685
amended_changed,373
with_amended_value,2202
revised_later,19168
nonreliance_8k,158


## 2. Detection at fixed budgets

Injected errors caught per budgeted flag, precision and recall on held-out
companies, averaged over replicates, with 95% company-bootstrap intervals.

In [3]:
metrics = pd.read_csv(tables / "budget_metrics.csv")
cols = [
    "budget_per_1000",
    "detector",
    "flags_budgeted",
    "flags_raised",
    "errors_per_budgeted_flag",
    "errors_per_budgeted_flag_lo",
    "errors_per_budgeted_flag_hi",
    "recall",
]
metrics[metrics["error_rate"] == config.ERROR_RATE][cols].round(3)

,budget_per_1000,detector,flags_budgeted,flags_raised,errors_per_budgeted_flag,errors_per_budgeted_flag_lo,errors_per_budgeted_flag_hi,recall
0,2,nothing,337.0,0.0,0.000,0.000,0.000,0.000
1,2,rules,337.0,337.0,0.396,0.378,0.409,0.079
2,2,plausibility,337.0,337.0,0.589,0.541,0.631,0.118
3,2,combined,337.0,337.0,0.396,0.378,0.409,0.079
4,5,nothing,843.0,0.0,0.000,0.000,0.000,0.000
5,5,rules,843.0,843.0,0.410,0.387,0.428,0.206
6,5,plausibility,843.0,843.0,0.406,0.377,0.436,0.204
7,5,combined,843.0,843.0,0.410,0.387,0.428,0.206
8,10,nothing,1687.0,0.0,0.000,0.000,0.000,0.000
9,10,rules,1687.0,1687.0,0.306,0.295,0.323,0.307


In [4]:
diffs = pd.read_csv(tables / "budget_differences.csv")
diffs[
    (diffs["error_rate"] == config.ERROR_RATE) & (diffs["metric"] == "errors_per_budgeted_flag")
].round(3)

,error_rate,replicates,budget_per_1000,comparison,metric,difference,lo,hi,share_resamples_positive
0,0.01,10,2,combined minus rules,errors_per_budgeted_flag,0.000,0.000,0.000,0.000
2,0.01,10,2,plausibility minus rules,errors_per_budgeted_flag,0.193,0.146,0.239,1.000
4,0.01,10,2,combined minus plausibility,errors_per_budgeted_flag,-0.193,-0.239,-0.146,0.000
6,0.01,10,5,combined minus rules,errors_per_budgeted_flag,0.000,0.000,0.000,0.000
8,0.01,10,5,plausibility minus rules,errors_per_budgeted_flag,-0.004,-0.039,0.032,0.431
10,0.01,10,5,combined minus plausibility,errors_per_budgeted_flag,0.004,-0.032,0.039,0.568
12,0.01,10,10,combined minus rules,errors_per_budgeted_flag,0.000,0.000,0.000,0.000
14,0.01,10,10,plausibility minus rules,errors_per_budgeted_flag,-0.026,-0.048,-0.012,0.000
16,0.01,10,10,combined minus plausibility,errors_per_budgeted_flag,0.026,0.012,0.048,1.000
18,0.01,10,20,combined minus rules,errors_per_budgeted_flag,0.029,0.000,0.052,0.867


The pre-specified primary comparison is *combined minus rules* at 10 flags per
1,000 cells. As the plan's deviations record, the identity rules alone raise
more flags than that budget allows, so the combined detector equals the rules
there by construction and the difference is zero. The informative comparisons
are at the smallest budget, where the plausibility layer alone is ahead, and at
the largest, where the combined detector adds the plausibility flags after the
rules run out.

In [5]:
by_type = pd.read_csv(tables / "recall_by_type.csv")
sel = by_type[
    (by_type["error_rate"] == config.ERROR_RATE)
    & (by_type["budget_per_1000"] == config.BUDGET_PRIMARY)
]
sel.pivot(index="error_type", columns="detector", values="recall").round(3)

detector,combined,nothing,plausibility,rules
error_type,,,,
copy_forward,0.098,0.0,0.004,0.098
cross_company_swap,0.323,0.0,0.353,0.323
digit_transposition,0.103,0.0,0.067,0.103
scale_slip,0.389,0.0,0.606,0.389
sign_flip,0.630,0.0,0.385,0.630


## 3. False-flag rate against target

Benjamini-Hochberg over conformal p-values, calibrated on clean development
companies, tested on held-out companies. The row calibrated on injected
development data is the pre-specified variant that flagged nothing.

In [6]:
ffr = pd.read_csv(tables / "false_flag_rate.csv")
ffr.round(4)

,calibration,target,detector,flags,false_flags,achieved_false_flag_rate,lo,hi,recall_of_errors,false_flags_in_periods_with_an_injected_cell,false_flag_rate_outside_those_periods,replicates
0,clean development data,0.05,plausibility,0.0,0.0,NaN,NaN,NaN,0.0000,0.0,NaN,10
1,clean development data,0.05,combined,3140.9,2485.0,0.7912,0.7492,0.8268,0.3561,1196.9,0.9931,10
2,clean development data,0.10,plausibility,0.0,0.0,NaN,NaN,NaN,0.0000,0.0,NaN,10
3,clean development data,0.10,combined,3140.9,2485.0,0.7912,0.7492,0.8268,0.3561,1196.9,0.9931,10
4,clean development data,0.20,plausibility,329.1,132.7,0.4011,0.3326,0.4629,0.1066,26.3,0.9989,10
5,clean development data,0.20,combined,3367.5,2611.4,0.7755,0.7339,0.8115,0.4105,1219.3,0.9935,10
6,injected development data (as pre-specified),0.05,plausibility,0.0,0.0,NaN,NaN,NaN,0.0000,0.0,NaN,1
7,injected development data (as pre-specified),0.05,combined,3118.0,2471.0,0.7925,0.7477,0.8310,0.3541,1163.0,0.9932,1
8,injected development data (as pre-specified),0.10,plausibility,0.0,0.0,NaN,NaN,NaN,0.0000,0.0,NaN,1
9,injected development data (as pre-specified),0.10,combined,3118.0,2471.0,0.7925,0.7477,0.8310,0.3541,1163.0,0.9932,1


## 4. Real amendments

Recall and enrichment of cells whose value was changed in a later 10-K/A or
10-Q/A, on the clean run.

In [7]:
amend = pd.read_csv(tables / "amendments.csv")
amend[
    [
        "budget_per_1000",
        "detector",
        "flags",
        "amended_among_flags",
        "amended_total",
        "recall_amended",
        "enrichment_amended",
        "enrichment_lo",
        "enrichment_hi",
        "enrichment_revised",
    ]
].round(3)

,budget_per_1000,detector,flags,amended_among_flags,amended_total,recall_amended,enrichment_amended,enrichment_lo,enrichment_hi,enrichment_revised
0,2,nothing,0.0,0.0,161.0,0.000,NaN,NaN,NaN,NaN
1,2,rules,337.0,0.0,161.0,0.000,0.000,0.0,23.446,1.545
2,2,plausibility,337.0,1.0,161.0,0.006,3.111,0.0,10.015,1.390
3,2,combined,337.0,0.0,161.0,0.000,0.000,0.0,23.446,1.545
4,5,nothing,0.0,0.0,161.0,0.000,NaN,NaN,NaN,NaN
5,5,rules,843.0,9.0,161.0,0.056,11.192,0.0,29.213,2.100
6,5,plausibility,843.0,2.0,161.0,0.012,2.487,0.0,7.697,1.359
7,5,combined,843.0,9.0,161.0,0.056,11.192,0.0,29.213,2.100
8,10,nothing,0.0,0.0,161.0,0.000,NaN,NaN,NaN,NaN
9,10,rules,1545.0,9.0,161.0,0.056,6.107,0.0,16.769,1.696


In [8]:
pd.read_csv(tables / "non_reliance.csv").round(4)

,company_periods,cells,flag_rate_combined_primary_budget,amended_rate
0,under_non_reliance,2512,0.0318,0.0561
1,other,166265,0.0097,0.0001


## 5. One flag, end to end

The first flag of the combined ranking in the primary replicate: its evidence
record, the template text, and the checker's view of that text.

In [9]:
cells = pd.read_parquet(results / "cells" / "primary_r00_cells.parquet")
evidence = [
    json.loads(line)
    for line in (results / "cells" / "primary_r00_evidence.jsonl")
    .read_text(encoding="utf-8")
    .splitlines()
]
first = cells.sort_values("rank_combined").query("rank_combined > 0").iloc[0]
record = next(r for r in evidence if first["cell_id"] in r["flagged_cell_ids"])
print(
    first[
        ["cell_id", "value_used", "injected", "error_type", "rules", "score", "p_value"]
    ].to_string()
)
print(json.dumps(record, indent=1)[:3000])

cell_id       1402436|2024-03-31|cost_of_revenue
value_used                           727800000.0
injected                                   False
error_type                                      
rules                                         R6
score                                   23.38096
p_value                                 0.002311
{
 "company": {
  "cik": 1402436,
  "name": "SS&C TECHNOLOGIES HOLDINGS, INC."
 },
 "computed": {
  "candidate_previous_period": 725700000.0,
  "candidate_ratio_to_revenue": 141261.21,
  "history_n": 11.0,
  "p_value": 0.002310924369747899,
  "prediction": 362920630.61,
  "residual": 364879369.39,
  "scale": 15605833.47,
  "score": 23.3809601
 },
 "detector": "plausibility",
 "flag_id": "P|1402436|2024-03-31|cost_of_revenue",
 "flagged_cell_ids": [
  "1402436|2024-03-31|cost_of_revenue"
 ],
 "period_end": "2024-03-31",
 "rule": null,
 "rule_statement": null,
 "sources": [
  {
   "accession_number": "0000950170-24-051251",
   "cell_id": "1402436|2024

In [10]:
text = template(record)
print(text)
check = check_text(text, record)
print("accepted:", check.accepted, " claims:", check.claims, " verified:", check.verified)

We refer to the Form 10-Q (accession number 0000950170-24-051251, filed May 1, 2024) for the period ended March 31, 2024. Cost of revenue of $727,800,000 for the 3 months then ended is $364,879,369.39 away from the $362,920,630.61 implied by the company's earlier filings and the other items of the period, more than the usual spread of $15,605,833.47 for this item. Please tell us how the figure was determined and whether it is correctly stated.
accepted: True  claims: 10  verified: 10


In [11]:
bad = text.replace("Please explain", "Net income was overstated by $5,000,000. Please explain")
check = check_text(bad, record)
print("accepted:", check.accepted)
for f in check.failures:
    print(" -", f)

accepted: True


## 6. Query text from the language model

Faithfulness of the generated texts: share of claims verified, rejection rate,
and the share of accepted texts that relied on rounding (strict pass).

In [12]:
faith_path = tables / "faithfulness.csv"
faith = pd.read_csv(faith_path) if faith_path.exists() else pd.DataFrame()
faith

,variant,run,texts,generation_errors,claims,claims_verified_share,texts_with_unverified_claim_share,rejection_rate,accepted,accepted_not_exact_share,model,backend
0,template,True,50,0,581,1.000000,0.00,0.00,50,0.0,NaN,template
1,llm_evidence,True,50,0,340,0.941176,0.40,0.40,30,0.1,llama3.2:3b,ollama
2,llm_raw,True,50,0,454,0.925110,0.36,0.36,32,0.0,llama3.2:3b,ollama


In [13]:
texts_path = results / "texts.jsonl"
if texts_path.exists():
    rows = [json.loads(line) for line in texts_path.read_text(encoding="utf-8").splitlines()]
    shown = 0
    for r in rows:
        if r["variant"] == "llm_evidence" and r["text"]:
            print(f"[{r['flag_id']}] accepted={r['accepted']}")
            print(r["text"])
            if r["failures"]:
                print("  rejected because:", r["failures"])
            print()
            shown += 1
            if shown == 3:
                break

[R6|1402436|2024-03-31] accepted=False
Dear SS&C Technologies Holdings, Inc.,

We observed that the gross profit value of $707,200,000.00 in your 10-Q filing (0000950170-24-051251) for the period ended March 31, 2024, is inconsistent with the calculated difference of $143,474,100.00, which suggests a discrepancy between the reported gross profit and the implied gross profit.

We would appreciate it if you could explain this discrepancy or revise your filing to reflect the correct values.

Sincerely,
[Your Name]
Securities Regulator
  rejected because: number $143,474,100.00 not in evidence

[R6|1402436|2024-03-31] accepted=False
Dear SS&C Technologies Holdings, Inc.,

We observed that the gross profit value of $707,200,000.00 in your 10-Q filing (0000950170-24-051251) for the period ended March 31, 2024, is inconsistent with the calculated difference of $143,474,100.00, which suggests a discrepancy between the reported gross profit and the implied gross profit.

We would appreciate it 

## 7. The audit log

The chain is verified and the tampering cases are listed.

In [14]:
log_path = results / "audit" / "audit.jsonl"
if log_path.exists():
    print(verify(log_path))
pd.read_csv(tables / "audit_tamper.csv")

ok: 224059 entries, chain intact


,tampering,detected,first_bad_entry,reason
0,edit a payload field,True,1.0,entry hash does not match its content
1,delete an entry,True,1.0,sequence number 2 where 1 was expected
2,insert an entry,True,2.0,previous hash does not match the preceding entry
3,swap two entries,True,1.0,sequence number 2 where 1 was expected
4,truncate the tail,True,223864.0,"223864 entries found, 223865 expected"
5,edit and recompute one hash,True,2.0,previous hash does not match the preceding entry
6,none (control),False,NaN,NaN


In [15]:
head = results / "audit_log_head.jsonl"
for line in head.read_text(encoding="utf-8").splitlines()[:3]:
    entry = json.loads(line)
    print(entry["seq"], entry["type"], entry["prev_hash"][:12], "->", entry["hash"][:12])

0 run_start 000000000000 -> e755b3f5f83f
1 flag e755b3f5f83f -> d3a0bfa77ddf
2 flag d3a0bfa77ddf -> cee2d84ce28c


## 8. Determinism and runtime

In [16]:
print(pd.read_csv(tables / "determinism.csv").to_string())
pd.read_csv(tables / "runtime.csv").round(1)

                    platform  python  quick                                              data_manifest_sha256  two_runs_identical  attempt                                                      flags_sha256                                                   evidence_sha256
0  Windows-11-10.0.26300-SP0  3.13.7  False  c6b69dd5dbf35dca7f23a6ab8defbc425cff6c2fd90b5a4858d17daf2b29e1f0                True        0  580ce5722bce9be27399fe0ea9c229569f17c9b8acbc83832e965af0192182ad  c2acd17964bfc58b01ac6f32933aeac561c583fa67619dc0e070b87fd9ce4a96


,tag,replicates,heldout_tested_cells,seconds_calibration,seconds_conformal,seconds_evidence,seconds_plausibility,seconds_rules,seconds_total,seconds_per_1000_heldout_cells
0,primary,10,168777.0,0.1,5.8,13.5,30.7,1.3,51.3,0.3
1,sens0.005,10,168777.0,0.1,7.1,19.2,44.9,2.0,73.4,0.4
2,sens0.02,10,168777.0,0.1,5.7,16.2,27.7,1.0,50.7,0.3
3,clean,1,168777.0,0.0,3.7,12.0,20.0,0.7,36.4,0.2
